# Sub1BitLLM - Colab Debug Notebook

Upload this zip and run to test the API.

In [ ]:
# Install dependencies
!pip install torch numpy --quiet

In [ ]:
import sys
sys.path.insert(0, '/content/sub1quant/src')

import torch
from Sub1BitLLM import Sub1BitLLM, Sub1BitConfig, from_fp16
from quantization import (
    ternary_quantize, ternary_pack, ternary_unpack,
    sigma_quantize, sigma_dequantize,
    quantize_factor, pack_factor, unpack_factor, dequantize_factor
)

print("=== Sub1BitLLM API Test ===")
print(f"Torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Test 1: Sub1BitConfig
config = Sub1BitConfig(
    codebook_dim=128,
    energy_threshold=0.95,
    rank=16,
    model_name="test-model"
)
print(f"Config: {config}")

In [ ]:
# Test 2: LowRankFactor forward and forward_lowrank
from Sub1BitLLM import LowRankFactor

U = torch.randn(4096, 16).half()
S = torch.randn(16).half()
Vt = torch.randn(16, 4096).half()

layer = LowRankFactor(U, S, Vt)
W = layer()
print(f"Reconstructed weight shape: {W.shape}")

x = torch.randn(1, 4096).half()
output = layer.forward_lowrank(x)
print(f"Lowrank output shape: {output.shape}")

output_full = torch.matmul(x, torch.matmul(U * S.unsqueeze(0), Vt))
error = (output - output_full).abs().max().item()
print(f"Max diff from full matmul: {error:.6e}")

In [ ]:
# Test 3: Sub1BitLLM state_dict and compression stats
model = Sub1BitLLM("dummy_path", config)
model.layers[0] = layer

state = model.state_dict()
print(f"State dict keys: {list(state.keys())}")
print(f"Compression stats: {model.compression_stats()}")

In [ ]:
# Test 4: Ternary quantize + pack roundtrip
print("=== Ternary Packing Roundtrip ===")

x = torch.randn(4096, 16)
q, scale = ternary_quantize(x)
packed = ternary_pack(q)
unpacked = ternary_unpack(packed, q.shape)

original_size = q.numel()
packed_size = packed.numel()
ratio = original_size / packed_size
print(f"Original: {original_size} int8 -> Packed: {packed_size} uint8 ({ratio:.1f}x)")
print(f"Values match: {(q == unpacked).all().item()}")
print(f"Effective bit-width: {8 / ratio:.2f} bits/value")

In [ ]:
# Test 5: Full quantize_factor + pack_factor + dequantize_factor roundtrip
print("=== Full Factor Quantization Roundtrip ===")

U = torch.randn(4096, 16)
S = torch.randn(16)
Vt = torch.randn(16, 4096)

q_data = quantize_factor(U, S, Vt)
p_data = pack_factor(q_data)
W_recon = dequantize_factor(p_data)

W_orig = torch.matmul(U * S.unsqueeze(0), Vt)
error = torch.norm(W_orig - W_recon) / torch.norm(W_orig)
print(f"Reconstruction error: {error:.6f}")
print(f"Quantized sizes: U={p_data['U_packed'].shape}, Vt={p_data['Vt_packed'].shape}, S={p_data['S'].shape}")

In [ ]:
# Test 6: GGUF export (uses shared GGUFWriter)
import os
os.makedirs("/content/sub1quant", exist_ok=True)

try:
    size = model.to_gguf("/content/sub1quant/test_model.gguf")
    print(f"GGUF exported: {size} bytes")
except Exception as e:
    print(f"GGUF export error: {e}")

In [ ]:
# Test 7: Load from checkpoints (will fail gracefully if no checkpoints)
try:
    model_loaded = from_fp16(
        "/content/models/llama-2-7b",
        config=config,
        checkpoint_dir="/content/sub1quant/checkpoints/",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    print(f"Loaded {len(model_loaded.layers)} layers")
except FileNotFoundError as e:
    print(f"No checkpoints found (expected): {e}")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

In [ ]:
# Test 8: Low-rank factorization import
from lowrank_factorization import low_rank_factorize, factorize_model_weights

print("=== Low-rank Factorization Test ===")
W = torch.randn(4096, 4096)
U, S, Vt, r = low_rank_factorize(W, energy_threshold=0.95)
print(f"Shape: {W.shape} -> Rank: {r}")
print(f"U: {U.shape}, S: {S.shape}, Vt: {Vt.shape}")
recon = torch.matmul(U * S.unsqueeze(0), Vt)
error = torch.norm(W - recon) / torch.norm(W)
print(f"Reconstruction error: {error:.6f}")

In [ ]:
print("\n=== All tests passed! ===")
print("\nNext steps:")
print("1. python quantize.py --model /path/to/model --output quantized/model.gguf")
print("2. python run_eval.py")
print("3. Or load checkpoints directly: from_fp16('...', checkpoint_dir='...')")